In [1]:
%load_ext autoreload
%autoreload 2

# Imports

In [2]:
from pathlib import Path
import shutil

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import open3d as o3d
from torch import Tensor, nn
import MinkowskiEngine as ME
import faiss
from tqdm.notebook import tqdm
from torch.profiler import profile, record_function, ProfilerActivity


from opr.models.place_recognition import MinkLoc3D, MinkLoc3Dv2
from opr.pipelines.place_recognition import PlaceRecognitionPipeline

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/usr/local/lib/python3.10/dist-packages/MinkowskiEngine-0.5.4-py3.10-linux-x86_64.egg/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(
2025-08-11 13:55:38.454 | WARNING  | opr.models.place_recognition.pointmamba:<module>:16 - The 'pointmamba' package is not installed. Please install it manually if neccessary.


# Constants

In [3]:
REPO_ROOT = Path("/home/docker_mmpr/multimodal-place-recognition")
DATASETS_ROOT = Path("/home/docker_mmpr/Datasets/")

SBER_OFFICE_DATA_DIR = DATASETS_ROOT / "2025-03-26-mmpr-datasets" / "mmpr_dataset_1" / "mav0"
assert SBER_OFFICE_DATA_DIR.exists(), f"Data directory {SBER_OFFICE_DATA_DIR} does not exist."


# DataReader definition

In [4]:
class DataReader:
    def __init__(
            self, csv_file: str | Path, lidar_scans_dir: str | Path, pointcloud_quantization_size: float = 0.1
        ) -> None:
        """Initialize DataReader for pose-timestamped point cloud data.

        Args:
            csv_file (str | Path): Path to the CSV file containing pose and timestamp data.
            lidar_scans_dir (str | Path): Directory containing the lidar scans in PCD format.
            pointcloud_quantization_size (float): Size for quantizing the point cloud coordinates.
                Default is 0.1.
        Raises:
            FileNotFoundError: If the CSV file or lidar scans directory does not exist.
        """
        csv_file = Path(csv_file)
        if not csv_file.exists():
            raise FileNotFoundError(f"CSV file {csv_file} does not exist.")

        self.lidar_scans_dir = Path(lidar_scans_dir)
        if not self.lidar_scans_dir.exists():
            raise FileNotFoundError(f"Lidar scans directory {self.lidar_scans_dir} does not exist.")

        self.df = self.read_csv(csv_file)

        self.scans_list = []
        for scan_id in self.df['lidar_timestamp'].values:
            path = self.lidar_scans_dir / f"{scan_id:018d}.pcd"
            if not path.exists():
                raise FileNotFoundError(f"Missing scan file: {path}")
            self.scans_list.append(path)

        self._pointcloud_quantization_size = pointcloud_quantization_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, Tensor]:
        """Get the pose and point cloud data for a given index.

        Args:
            idx (int): Index of the data point to retrieve.
        Returns:
            dict: A dictionary containing:
                - pose (Tensor): The pose as a 7-element tensor [x, y, z, qx, qy, qz, qw].
                - pointcloud_lidar_coords (Tensor): The coordinates of the point cloud as an Nx3 tensor.
                - pointcloud_lidar_feats (Tensor): The features of the point cloud as an Nx1 tensor (intensity or ones).
        Raises:
            IndexError: If the index is out of range.
            ValueError: If the scan file is empty or has an unexpected format.
        """
        if idx < 0 or idx >= len(self.df):
            raise IndexError("Index out of range.")

        pose = self.df[["x", "y", "z", "qx", "qy", "qz", "qw"]].iloc[idx].to_numpy()
        scan_filepath = self.scans_list[idx]
        pc_coords, pc_feats = self.read_scan(scan_filepath)

        output_dict = {
            "pose": Tensor(pose),
            "pointcloud_lidar_coords": Tensor(pc_coords),
            "pointcloud_lidar_feats": Tensor(pc_feats)
        }

        return output_dict

    def collate_fn(self, batch: list[dict[str, Tensor]]) -> dict[str, Tensor]:
        """Collate function to combine a batch of data points into a single dictionary.
        Args:
            batch (list[dict[str, Tensor]]): A list of dictionaries containing pose and point cloud data.
        Returns:
            dict: A dictionary containing:
                - poses (Tensor): Poses as an Nx7 tensor.
                - pointclouds_lidar_coords (Tensor): Point cloud coordinates as an Nx3 tensor.
                - pointclouds_lidar_feats (Tensor): Point cloud features as an Nx1 tensor.
        """
        poses = torch.stack([item['pose'] for item in batch])

        coords_list = [e["pointcloud_lidar_coords"] for e in batch]
        feats_list = [e["pointcloud_lidar_feats"] for e in batch]
        quantized_coords_list = []
        quantized_feats_list = []
        for coords, feats in zip(coords_list, feats_list):
            quantized_coords, quantized_feats = ME.utils.sparse_quantize(
                coordinates=coords,
                features=feats,
                quantization_size=self._pointcloud_quantization_size,
            )
            quantized_coords_list.append(quantized_coords)
            quantized_feats_list.append(quantized_feats)

        return {
            "poses": poses,
            "pointclouds_lidar_coords": ME.utils.batched_coordinates(quantized_coords_list),
            "pointclouds_lidar_feats": torch.cat(quantized_feats_list)
        }

    def read_scan(self, scan_filepath: str | Path) -> tuple[np.ndarray, np.ndarray]:
        """Read a point cloud scan from a file.
        Args:
            scan_filepath (str | Path): Path to the point cloud file.
        Returns:
            tuple: A tuple containing:
                - coordinates (np.ndarray): The coordinates of the point cloud as an Nx3 array.
                - features (np.ndarray): The features of the point cloud as an Nx1 array (intensity or ones).
        Raises:
            ValueError: If the scan file is empty or has an unexpected format.
        """
        scan = o3d.io.read_point_cloud(str(scan_filepath))
        if not scan.has_points():
            raise ValueError(f"Scan file {scan_filepath} is empty or invalid.")
        # Convert to numpy array for easier manipulation
        scan = np.asarray(scan.points)
        coordinates = scan[:, :3]  # Get the first three columns (x, y, z)
        if scan.shape[1] == 3:
            features = np.ones((coordinates.shape[0], 1))
        elif scan.shape[1] == 4:
            features = scan[:, 3:4]  # Get the fourth column (intensity)
        else:
            raise ValueError(f"Unexpected scan format with shape {scan.shape}. Expected 3 or 4 columns.")
        return coordinates, features

    def read_csv(self, filepath: str | Path) -> pd.DataFrame:
        """Read a CSV file containing pose and timestamp data.
        Args:
            filepath (str | Path): Path to the CSV file.
        Returns:
            pd.DataFrame: A DataFrame containing the pose and timestamp data.
        Raises:
            FileNotFoundError: If the CSV file does not exist.
        """
        dtype_mapping = {
            'pose_timestamp': np.int64,
            'lidar_timestamp': np.int64,
            'x': np.float64,
            'y': np.float64,
            'z': np.float64,
            'qx': np.float64,
            'qy': np.float64,
            'qz': np.float64,
            'qw': np.float64,
        }
        df = pd.read_csv(filepath, dtype=dtype_mapping)
        return df


# Init datareaders

In [5]:
PC_QUANTIZATION_SIZE = 0.05

# Database: batched processing for efficient descriptor extraction
database_reader = DataReader(
    csv_file=SBER_OFFICE_DATA_DIR / "db_lidar_frames.csv",
    lidar_scans_dir=SBER_OFFICE_DATA_DIR / "lidar_joined" / "data",
    pointcloud_quantization_size=PC_QUANTIZATION_SIZE,
)

database_dl = torch.utils.data.DataLoader(
    database_reader,
    batch_size=1,
    shuffle=False,
    collate_fn=database_reader.collate_fn,
    num_workers=4,
    pin_memory=True,
    drop_last=False,
)

# Compute inference time for MinkLoc3Dv1/v2

## Init models

In [6]:
weights_v1 = torch.load(REPO_ROOT / "data" / "checkpoints" / "minkloc3d_nclt.pth")
weights_v2 = torch.load(REPO_ROOT / "data" / "checkpoints" / "minkloc3dv2_nclt.pth")

model_v1 = MinkLoc3D()
model_v1.load_state_dict(weights_v1, strict=False)
model_v1.eval()
if torch.cuda.is_available():
    model_v1 = model_v1.cuda()
else:
    print("CUDA is not available, running MinkLoc3Dv1 on CPU.")


model_v2 = MinkLoc3Dv2()
model_v2.load_state_dict(weights_v2, strict=False)
model_v2.eval()
if torch.cuda.is_available():
    model_v2 = model_v2.cuda()
else:
    print("CUDA is not available, running MinkLoc3Dv2 on CPU.")

In [7]:
def compute_inference_time(model, input, runs=100, warmup=10):
    model.eval()
    gpu_model_times = []

    # Warm-up to initialize 
    with torch.no_grad():
        for _ in range(warmup):
            _ = model(input)
    
    # Record elapsed time for every run
    with torch.no_grad():
        for _ in tqdm(range(runs)):
            torch.cuda.synchronize()
            start_event = torch.cuda.Event(enable_timing=True)
            end_event = torch.cuda.Event(enable_timing=True)
            
            start_event.record()
            _ = model(input)
            end_event.record()
            torch.cuda.synchronize()

            elapsed_time_ms = start_event.elapsed_time(end_event)
            gpu_model_times.append(elapsed_time_ms)

    return {
        "mean": np.mean(gpu_model_times),
        "std": np.std(gpu_model_times)
    }


In [8]:
batch = next(iter(database_dl))
batch = {k: v.to("cuda") for k, v in batch.items()}
batch['pointclouds_lidar_coords'].shape

torch.Size([13207, 4])

### MinkLoc3Dv1 inference time

`torch.profiler` shows execution time per operator. 

In [11]:
with torch.no_grad():
    with profile(activities=[ProfilerActivity.CUDA], record_shapes=True) as prof:
        with record_function("model_inference"):
            model_v1(batch)     
print(prof.key_averages().table(sort_by='cuda_time_total'))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
void minkowski::detail::matmul<float, unsigned int, ...         0.00%       0.000us         0.00%       0.000us       0.000us     481.000us        42.42%     481.000us       2.586us           186  
void minkowski::detail::matmul<float, unsigned int, ...         0.00%       0.000us         0.00%       0.000us       0.000us     140.000us        12.35%     140.000us       1.120us           125  
void cub:

STAGE:2025-08-11 13:57:25 591665:591665 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2025-08-11 13:57:25 591665:591665 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2025-08-11 13:57:25 591665:591665 ActivityProfilerController.cpp:322] Completed Stage: Post Processing


In [12]:
results_v1 = compute_inference_time(model_v1, batch)
mean_v1, std_v1 = results_v1["mean"], results_v1["std"]
print("GPU Model Time (ms):", mean_v1)
print("GPU Model Time Std (ms):", std_v1)

  0%|          | 0/100 [00:00<?, ?it/s]

GPU Model Time (ms): 4.009998109340668
GPU Model Time Std (ms): 0.09169054485141485


### MinkLoc3Dv2 inference time

In [13]:
with torch.no_grad():
    with profile(activities=[ProfilerActivity.CUDA], record_shapes=True) as prof:
        with record_function("model_inference"):
            model_v2(batch)     
print(prof.key_averages().table(sort_by='cuda_time_total'))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
void minkowski::detail::matmul<float, unsigned int, ...         0.00%       0.000us         0.00%       0.000us       0.000us     708.000us        32.98%     708.000us       3.806us           186  
                        ampere_sgemm_32x32_sliced1x4_nn         0.00%       0.000us         0.00%       0.000us       0.000us     226.000us        10.53%     226.000us       3.965us            57  
void cub:

STAGE:2025-08-11 13:57:29 591665:591665 ActivityProfilerController.cpp:312] Completed Stage: Warm Up
STAGE:2025-08-11 13:57:30 591665:591665 ActivityProfilerController.cpp:318] Completed Stage: Collection
STAGE:2025-08-11 13:57:30 591665:591665 ActivityProfilerController.cpp:322] Completed Stage: Post Processing


In [14]:
results_v2 = compute_inference_time(model_v2, batch)
mean_v2, std_v2 = results_v2["mean"], results_v2["std"]
print("GPU Model Time (ms):", mean_v2)
print("GPU Model Time Std (ms):", std_v2)

  0%|          | 0/100 [00:00<?, ?it/s]

GPU Model Time (ms): 11.132483501434326
GPU Model Time Std (ms): 2.7881993641388325
